# Lab03 – Data Setup (Chạy trước lab3_solution.ipynb)

File này **độc lập** với Lab01 và Lab02. Mục đích:
- Download MovieLens dataset từ Kaggle
- Tạo Kafka topics: `Lab1_movies`, `Lab1_ratings`, `Lab1_tags`
- Push toàn bộ dữ liệu vào Kafka một lần
- Verify dữ liệu đã có trong Kafka

**Yêu cầu trước khi chạy:**
- Docker đang chạy: `docker compose up -d` (từ thư mục gốc `LAB_BigData/`)
- Virtual env đã kích hoạt với: `pyspark==4.0.1 kagglehub confluent-kafka`

In [5]:
from pyspark.sql import SparkSession
import sys

KAFKA_BROKERS = "localhost:9092,localhost:9192,localhost:9292"

# SparkSession chỉ cần Kafka, không cần GraphFrames ở bước này
spark = (SparkSession.builder
    .appName("Lab03_DataSetup")
    .master("local[*]")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "4g")
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1,"
            "org.apache.kafka:kafka-clients:3.6.0,"
            "org.apache.spark:spark-streaming-kafka-0-10_2.13:4.0.1")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark version: {spark.version}")

✅ Spark version: 4.0.1


In [6]:
import kagglehub

# Download dataset MovieLens từ Kaggle
print("📥 Đang download MovieLens dataset...")
path = kagglehub.dataset_download("grouplens/movielens-latest-small")
print(f"✅ Dataset đã có tại: {path}")

# Đọc CSV vào Spark DataFrame
df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies  = spark.read.csv(path + "/movies.csv",  header=True, inferSchema=True)
df_tags    = spark.read.csv(path + "/tags.csv",    header=True, inferSchema=True)

print(f"Ratings: {df_ratings.count():,} dòng")
print(f"Movies : {df_movies.count():,} dòng")
print(f"Tags   : {df_tags.count():,} dòng")

df_ratings.show(2)
df_movies.show(2)
df_tags.show(2)

📥 Đang download MovieLens dataset...
✅ Dataset đã có tại: /home/hkqnva/.cache/kagglehub/datasets/grouplens/movielens-latest-small/versions/2
Ratings: 100,836 dòng
Movies : 9,742 dòng
Tags   : 3,683 dòng
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
+------+-------+------+---------+
only showing top 2 rows
+-------+----------------+--------------------+
|movieId|           title|              genres|
+-------+----------------+--------------------+
|      1|Toy Story (1995)|Adventure|Animati...|
|      2|  Jumanji (1995)|Adventure|Childre...|
+-------+----------------+--------------------+
only showing top 2 rows
+------+-------+---------------+----------+
|userId|movieId|            tag| timestamp|
+------+-------+---------------+----------+
|     2|  60756|          funny|1445714994|
|     2|  60756|Highly quotable|1445714996|
+------+-------+---------------+-----

In [11]:
from confluent_kafka.admin import AdminClient, NewTopic

admin_client = AdminClient({'bootstrap.servers': KAFKA_BROKERS})
TOPICS = ["Lab1_ratings", "Lab1_movies", "Lab1_tags"]

# --- Xóa topics cũ nếu tồn tại ---
print("🗑️  Đang xóa topics cũ (nếu có)...")
fs_delete = admin_client.delete_topics(TOPICS)
for topic, f in fs_delete.items():
    try:
        f.result()
        print(f"  Đã xóa: '{topic}'")
    except Exception as e:
        print(f"  (Bỏ qua, không tìm thấy topic '{topic}': {e})")

# Chờ một chút để Kafka xử lý việc xóa
import time
time.sleep(3)

# --- Tạo topics mới ---
print("\n📦 Đang tạo topics mới...")
new_topics = [
    NewTopic(topic="Lab1_ratings", num_partitions=1, replication_factor=3),
    NewTopic(topic="Lab1_movies",  num_partitions=1, replication_factor=3),
    NewTopic(topic="Lab1_tags",    num_partitions=1, replication_factor=3),
]
fs_create = admin_client.create_topics(new_topics)
for topic, f in fs_create.items():
    try:
        f.result()
        print(f"  ✅ Tạo topic thành công: '{topic}'")
    except Exception as e:
        print(f"  ❌ Lỗi khi tạo topic '{topic}': {e}")

🗑️  Đang xóa topics cũ (nếu có)...
  Đã xóa: 'Lab1_ratings'
  Đã xóa: 'Lab1_movies'
  Đã xóa: 'Lab1_tags'

📦 Đang tạo topics mới...
  ✅ Tạo topic thành công: 'Lab1_ratings'
  ✅ Tạo topic thành công: 'Lab1_movies'
  ✅ Tạo topic thành công: 'Lab1_tags'


In [13]:
# Push toàn bộ dữ liệu vào Kafka một lần (không chia batch)
# Format: mỗi dòng CSV được chuyển thành JSON string làm Kafka value

def push_to_kafka(df, topic_name):
    """Push một Spark DataFrame vào Kafka topic theo format JSON."""
    print(f"📤 Đang push '{topic_name}'...")
    (
        df.selectExpr("to_json(struct(*)) AS value")
          .write
          .format("kafka")
          .option("kafka.bootstrap.servers", KAFKA_BROKERS)
          .option("topic", topic_name)
          .save()
    )
    print(f"  ✅ Xong! ({df.count():,} records)")

push_to_kafka(df_ratings, "Lab1_ratings")
push_to_kafka(df_movies,  "Lab1_movies")
push_to_kafka(df_tags,    "Lab1_tags")

print("\n🎉 Đã push toàn bộ dữ liệu vào Kafka thành công!")

📤 Đang push 'Lab1_ratings'...
  ✅ Xong! (100,836 records)
📤 Đang push 'Lab1_movies'...
  ✅ Xong! (9,742 records)
📤 Đang push 'Lab1_tags'...
  ✅ Xong! (3,683 records)

🎉 Đã push toàn bộ dữ liệu vào Kafka thành công!


In [15]:
# Verify: đọc lại từ Kafka để xác nhận data đã có
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, LongType, DoubleType

movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title",   StringType(),  True),
    StructField("genres",  StringType(),  True),
])
ratings_schema = StructType([
    StructField("userId",    IntegerType(), True),
    StructField("movieId",   IntegerType(), True),
    StructField("rating",    DoubleType(),  True),
    StructField("timestamp", LongType(),    True),
])
tags_schema = StructType([
    StructField("userId",    IntegerType(), True),
    StructField("movieId",   IntegerType(), True),
    StructField("tag",       StringType(),  True),
    StructField("timestamp", LongType(),    True),
])

def read_from_kafka(topic, schema):
    return (
        spark.read.format("kafka")
            .option("kafka.bootstrap.servers", KAFKA_BROKERS)
            .option("subscribe", topic)
            .option("startingOffsets", "earliest")
            .load()
            .selectExpr("CAST(value AS STRING) as json_str")
            .select(from_json(col("json_str"), schema).alias("data"))
            .select("data.*")
    )

v_movies  = read_from_kafka("Lab1_movies",  movies_schema)
v_ratings = read_from_kafka("Lab1_ratings", ratings_schema)
v_tags    = read_from_kafka("Lab1_tags",    tags_schema)

print("=== Verify dữ liệu trong Kafka ===")
print(f"  Lab1_movies : {v_movies.count():,} records")
print(f"  Lab1_ratings: {v_ratings.count():,} records")
print(f"  Lab1_tags   : {v_tags.count():,} records")

print("\nSample movies:")
v_movies.show(3, truncate=False)
print("Sample ratings:")
v_ratings.show(3)

print("\n✅ Setup hoàn tất! Bạn có thể mở lab3_solution.ipynb để làm bài.")

=== Verify dữ liệu trong Kafka ===
  Lab1_movies : 19,484 records


  Lab1_ratings: 201,672 records
  Lab1_tags   : 7,366 records

Sample movies:
+-------+-----------------------+-------------------------------------------+
|movieId|title                  |genres                                     |
+-------+-----------------------+-------------------------------------------+
|1      |Toy Story (1995)       |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)         |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)|Comedy|Romance                             |
+-------+-----------------------+-------------------------------------------+
only showing top 3 rows
Sample ratings:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
+------+-------+------+---------+
only showing top 3 rows

✅ Setup hoàn tất! Bạn có thể mở lab3_solution.ipynb để làm bài.
